# 02 - Feature Engineering y Partición de Datos

Después de haber limpiado el dataset en el EDA, el siguiente paso es preparar los datos (*Feature Engineering*) para que los algoritmos de Machine Learning puedan procesarlos.

El objetivo principal de este TFM es predecir la clasificación completa de un ticket en el CRM. Por lo tanto, no vamos a predecir solo la cola, sino la combinación de la "tripleta" completa. 

En este notebook haremos tres cosas:
1. **Crear la variable objetivo (Target):** Concatenar `queue`, `type` y `priority` en una única columna.
2. **Crear la variable predictora (Feature):** Unir el `subject` y el `body` para darle todo el contexto posible al modelo.
3. **Train/Test Split:** Dividir el dataset dejando un 80% para entrenar y un 20% para evaluar. Usaremos partición estratificada (`stratify`) para asegurar que las clases menos frecuentes mantengan su proporción y no desaparezcan en el conjunto de test.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# 1. Cargamos el dataset limpio del EDA
df = pd.read_parquet("../data/processed/df_final_silver.parquet")
print(f"Total de tickets provenientes del EDA: {len(df)}")

# Nos quedamos de momento solo con los tickets en inglés para el entrenamiento base
df_en = df[df['language'] == 'en'].copy()
print(f"Tickets en inglés disponibles: {len(df_en)}")

# 2. Ingeniería de la variable objetivo (Y)
# Juntamos las 3 variables del CRM para crear la etiqueta final a predecir
df_en['target'] = df_en['queue'] + " - " + df_en['type'] + " - " + df_en['priority']
num_clases = df_en['target'].nunique()
print(f"Número de combinaciones (clases) generadas: {num_clases}")

# 3. Ingeniería de la variable predictora (X)
# Juntamos el asunto y el cuerpo en un solo texto
df_en['texto_completo'] = df_en['subject'] + " " + df_en['body']

# Filtramos para quedarnos solo con las dos columnas que irán al modelo
df_ml = df_en[['texto_completo', 'target']].dropna()

# 4. Partición Train / Test
X = df_ml['texto_completo']
y = df_ml['target']

# Es vital usar stratify=y para no romper la distribución de las 84 clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nReparto de datos:")
print(f"Train: {len(X_train)} tickets")
print(f"Test: {len(X_test)} tickets")

# 5. Guardado físico
ruta_features = "../data/features/"
os.makedirs(ruta_features, exist_ok=True)

# Guardamos los textos y los targets en archivos separados
X_train.to_csv(ruta_features + "en_X_train_text.csv", index=False)
X_test.to_csv(ruta_features + "en_X_test_text.csv", index=False)
y_train.to_csv(ruta_features + "en_y_train.csv", index=False)
y_test.to_csv(ruta_features + "en_y_test.csv", index=False)

print("\nArchivos guardados correctamente en data/features/")

Total de tickets provenientes del EDA: 23867
Tickets en inglés disponibles: 23117
Número de combinaciones (clases) generadas: 84

Reparto de datos:
Train: 18493 tickets
Test: 4624 tickets

Archivos guardados correctamente en data/features/
